# 直接计算 MeTp 的 T_SR（D 发射 → C 接收）

本 notebook 按你的最新要求重整为 **MeTp 的 T_SR**，过程为 **M 星发射至 T 星**，并采用：

- **T 星 → C 星（接收端）**
- **M 星 → D 星（发射端）**

也就是在程序中使用：
- `r_M, \dot r_M, \ddot r_M` 来自 **D 星**
- `r_T, \dot r_T, \ddot r_T` 来自 **C 星**

## 本版修改要点

1. **卫星映射修改**
   - 原先的 `T` 改为 `C`
   - 原先的 `M` 改为 `D`

2. **加速度计算方式修改**
   不再用速度数值微分计算加速度，而改为地球点质量模型：

   \[
   a = \frac{GM}{R^2}
   \]

   在程序中采用等价的向量形式：

   \[
   \mathbf a = -\frac{\mu}{R^3}\mathbf r, \qquad \mu = GM
   \]

   其中：
   - \( \mu = GM \) 为地球引力常数
   - \( R = |\mathbf r| \) 为卫星到地心的距离

3. **T_SR 公式结构保持原程序写法**
   - 仍读取：
     - `LightTime_T_GR_MeTp.xlsx`
     - `LightTime_T_GR_TpMr.xlsx`
   - 仍输出 Excel 文件 `.xlsx`

## 本版几何约定

在同一个 `gps_time` 上评估：

\[
d_0 = \frac{r_M-r_T}{|r_M-r_T|}
    = \frac{r_D-r_C}{|r_D-r_C|}
\]

即时几何光行时：

\[
\Delta t_{\rm inst} = \frac{|r_D-r_C|}{c_0}
\]

这里对应 **MeTp：D 发射，C 接收**。



In [13]:
import re
import numpy as np
import pandas as pd

# ----------------------------
# constants
# ----------------------------
C0 = 299792458.0                  # speed of light [m/s]
MU_EARTH = 3.986004418e14         # GM of Earth [m^3/s^2]

# ----------------------------
# input / output files
# ----------------------------
GNI_C_PATH = "GNI1B_2022-06-05_C_04.txt"      # C = T (receiver)
GNI_D_PATH = "GNI1B_2022-06-05_D_04.txt"      # D = M (emitter)

CSV_GR_METP = "LightTime_T_GR_MeTp.xlsx"      # must contain columns: gps_time, delta_t_s
CSV_GR_TPMR = "LightTime_T_GR_TpMr.xlsx"      # must contain columns: gps_time, delta_t_s

OUT_XLSX = "T_SR.xlsx"


In [14]:
def find_first_data_row(filepath: str) -> int:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if re.match(r"^\s*\d+\s+", line):
                return i
    raise RuntimeError(f"No data rows found in {filepath}")


def read_gni1b(filepath: str, expected_sat: str) -> pd.DataFrame:
    cols = [
        "gps_time", "sat_id", "coord_ref",
        "x", "y", "z",
        "xerr", "yerr", "zerr",
        "vx", "vy", "vz",
        "vxerr", "vyerr", "vzerr",
        "qualflg",
    ]
    skip = find_first_data_row(filepath)
    df = pd.read_csv(filepath, sep=r"\s+", header=None, names=cols, skiprows=skip)
    df = df[(df["sat_id"] == expected_sat) & (df["coord_ref"] == "I")].copy()
    df = df[["gps_time", "x", "y", "z", "vx", "vy", "vz", "qualflg"]]
    df["gps_time"] = df["gps_time"].astype(np.int64)
    return df.sort_values("gps_time").reset_index(drop=True)


def read_tgr(filepath: str, out_name: str) -> pd.DataFrame:
    df = pd.read_excel(filepath)
    if "gps_time" not in df.columns or "delta_t_s" not in df.columns:
        raise ValueError(
            f"{filepath} must contain columns ['gps_time', 'delta_t_s'], "
            f"but got: {list(df.columns)}"
        )
    out = df[["gps_time", "delta_t_s"]].copy()
    out["gps_time"] = out["gps_time"].astype(np.int64)
    out = out.rename(columns={"delta_t_s": out_name})
    return out.sort_values("gps_time").reset_index(drop=True)


def gravitational_acceleration(r_xyz: np.ndarray, mu: float = MU_EARTH) -> np.ndarray:
    """
    Earth point-mass gravity:
        a = -mu / |r|^3 * r
    whose magnitude is:
        |a| = mu / |r|^2 = GM / R^2
    """
    r_xyz = np.asarray(r_xyz, dtype=float)
    r_norm = np.linalg.norm(r_xyz)
    if r_norm == 0.0:
        raise ValueError("Position norm is zero; cannot compute gravity acceleration.")
    return -(mu / r_norm**3) * r_xyz


def add_gravity_acceleration(df: pd.DataFrame, sat_tag: str) -> pd.DataFrame:
    out = df.copy()
    r_all = out[["x", "y", "z"]].to_numpy(dtype=float)
    a_all = np.apply_along_axis(gravitational_acceleration, 1, r_all)
    out["ax"] = a_all[:, 0]
    out["ay"] = a_all[:, 1]
    out["az"] = a_all[:, 2]
    out["a_mag"] = np.linalg.norm(a_all, axis=1)

    rename_map = {
        "x": f"x_{sat_tag}",
        "y": f"y_{sat_tag}",
        "z": f"z_{sat_tag}",
        "vx": f"vx_{sat_tag}",
        "vy": f"vy_{sat_tag}",
        "vz": f"vz_{sat_tag}",
        "ax": f"ax_{sat_tag}",
        "ay": f"ay_{sat_tag}",
        "az": f"az_{sat_tag}",
        "a_mag": f"a_mag_{sat_tag}",
        "qualflg": f"qualflg_{sat_tag}",
    }
    return out.rename(columns=rename_map)


def norm_rows(arr: np.ndarray) -> np.ndarray:
    return np.linalg.norm(arr, axis=1)


def dot_rows(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return np.einsum("ij,ij->i", a, b)


In [15]:
# ----------------------------
# read and align data
# ----------------------------
df_C = read_gni1b(GNI_C_PATH, expected_sat="C")   # C = T (receiver)
df_D = read_gni1b(GNI_D_PATH, expected_sat="D")   # D = M (emitter)

df_C = add_gravity_acceleration(df_C, sat_tag="C")
df_D = add_gravity_acceleration(df_D, sat_tag="D")

gr_metp = read_tgr(CSV_GR_METP, out_name="T_GR_MeTp_s")
gr_tpmr = read_tgr(CSV_GR_TPMR, out_name="T_GR_TpMr_s")

df = (
    df_C.merge(df_D, on="gps_time", how="inner")
        .merge(gr_metp, on="gps_time", how="inner")
        .merge(gr_tpmr, on="gps_time", how="inner")
        .sort_values("gps_time")
        .reset_index(drop=True)
)

print("rows =", len(df))
if len(df) > 0:
    print("gps_time range:", int(df["gps_time"].iloc[0]), "->", int(df["gps_time"].iloc[-1]))
df.head()


rows = 86400
gps_time range: 707659200 -> 707745599


,gps_time,x_C,y_C,z_C,vx_C,vy_C,vz_C,qualflg_C,ax_C,ay_C,...,vx_D,vy_D,vz_D,qualflg_D,ax_D,ay_D,az_D,a_mag_D,T_GR_MeTp_s,T_GR_TpMr_s
0,707659200,3.716399e+06,3.208044e+06,4.780815e+06,-4149.229192,-3352.914250,5461.061418,10000000,-4.603380,-3.973698,...,-4030.053419,-3250.486988,5610.213439,10000000,-4.732024,-4.077941,-5.746020,8.487544,8.421256e-13,8.420871e-13
1,707659201,3.712248e+06,3.204689e+06,4.786273e+06,-4153.820680,-3356.877897,5455.131498,10000000,-4.598258,-3.969559,...,-4034.774291,-3254.555538,5604.458237,10000000,-4.727052,-4.073931,-5.752991,8.487570,8.421283e-13,8.420898e-13
2,707659202,3.708092e+06,3.201331e+06,4.791725e+06,-4158.407018,-3360.837382,5449.194835,10000000,-4.593130,-3.965416,...,-4039.490160,-3258.620051,5598.696106,10000000,-4.722073,-4.069917,-5.759955,8.487596,8.421310e-13,8.420925e-13
3,707659203,3.703931e+06,3.197968e+06,4.797171e+06,-4162.988200,-3364.792697,5443.251438,10000000,-4.587996,-3.961268,...,-4044.201020,-3262.680523,5592.927053,10000000,-4.717089,-4.065897,-5.766912,8.487622,8.421336e-13,8.420952e-13
4,707659204,3.699766e+06,3.194601e+06,4.802612e+06,-4167.564221,-3368.743840,5437.301313,10000000,-4.582856,-3.957114,...,-4048.906867,-3266.736949,5587.151084,10000000,-4.712099,-4.061872,-5.773862,8.487648,8.421363e-13,8.420979e-13


In [16]:
def compute_tsr_metp(df: pd.DataFrame) -> pd.DataFrame:
    """
    Compute MeTp T_SR with mapping:
        M -> D satellite (emitter)
        T -> C satellite (receiver)

    Therefore:
        r_M, v_M, a_M come from D
        r_T, v_T, a_T come from C
    """
    # M = D (emitter)
    rM = df[["x_D", "y_D", "z_D"]].to_numpy(dtype=float)
    vM = df[["vx_D", "vy_D", "vz_D"]].to_numpy(dtype=float)
    aM = df[["ax_D", "ay_D", "az_D"]].to_numpy(dtype=float)

    # T = C (receiver)
    rT = df[["x_C", "y_C", "z_C"]].to_numpy(dtype=float)
    vT = df[["vx_C", "vy_C", "vz_C"]].to_numpy(dtype=float)
    aT = df[["ax_C", "ay_C", "az_C"]].to_numpy(dtype=float)

    dr = rM - rT
    rho = norm_rows(dr)
    if np.any(rho == 0.0):
        raise ValueError("Found zero separation between M and T; cannot define d0.")

    d0 = dr / rho[:, None]
    dt_inst = rho / C0

    d0vM = dot_rows(d0, vM)
    d0vT = dot_rows(d0, vT)
    d0aM = dot_rows(d0, aM)
    d0aT = dot_rows(d0, aT)

    vM2 = dot_rows(vM, vM)
    vT2 = dot_rows(vT, vT)
    vTvM = dot_rows(vT, vM)
    aMvM = dot_rows(aM, vM)
    aTvM = dot_rows(aT, vM)
    vTaT = dot_rows(vT, aT)

    vT_minus_2vM = vT - 2.0 * vM
    norm_vT_minus_2vM_sq = dot_rows(vT_minus_2vM, vT_minus_2vM)

    T_GR_MeTp = df["T_GR_MeTp_s"].to_numpy(dtype=float)
    T_GR_TpMr = df["T_GR_TpMr_s"].to_numpy(dtype=float)

    term1 = (
        dt_inst * (d0vT - 2.0 * d0vM)
        + dt_inst**2 * (2.0 * d0aM - 0.5 * d0aT)
    ) / C0

    term2 = (
        dt_inst * (norm_vT_minus_2vM_sq + d0vT**2)
    ) / (2.0 * C0**2)

    term3 = (
        dt_inst**2 * (
            -2.0 * d0aM * (d0vM - d0vT)
            -4.0 * aMvM
            +2.0 * aTvM
            -d0aT * d0vT
            +aTvM
            -0.5 * vTaT
        )
    ) / (C0**2)

    term4 = (
        dt_inst * (
            d0vT * (2.0 * vM2 + vT2 - 2.0 * vTvM)
            -2.0 * vM2 * d0vM
        )
    ) / (C0**3)

    term5 = (
        T_GR_TpMr * d0vT
        - (T_GR_TpMr + T_GR_MeTp) * d0vM
    ) / C0

    T_SR = term1 + term2 + term3 + term4 + term5
    rho_SR = T_SR * C0

    out = df[["gps_time"]].copy()
    out["dt_inst_s"] = dt_inst
    out["d0_x"] = d0[:, 0]
    out["d0_y"] = d0[:, 1]
    out["d0_z"] = d0[:, 2]

    out["T_GR_MeTp_s"] = T_GR_MeTp
    out["T_GR_TpMr_s"] = T_GR_TpMr

    out["term1_s"] = term1
    out["term2_s"] = term2
    out["term3_s"] = term3
    out["term4_s"] = term4
    out["term5_s"] = term5

    out["T_SR_s"] = T_SR
    out["rho_SR_m"] = rho_SR

    out["d0_dot_vD_mps"] = d0vM
    out["d0_dot_vC_mps"] = d0vT
    out["d0_dot_aD_mps2"] = d0aM
    out["d0_dot_aC_mps2"] = d0aT

    out["aD_x_mps2"] = aM[:, 0]
    out["aD_y_mps2"] = aM[:, 1]
    out["aD_z_mps2"] = aM[:, 2]
    out["aD_mag_mps2"] = df["a_mag_D"].to_numpy(dtype=float)

    out["aC_x_mps2"] = aT[:, 0]
    out["aC_y_mps2"] = aT[:, 1]
    out["aC_z_mps2"] = aT[:, 2]
    out["aC_mag_mps2"] = df["a_mag_C"].to_numpy(dtype=float)

    return out


In [17]:
# ----------------------------
# run and export
# ----------------------------
result = compute_tsr_metp(df)
result.to_excel(OUT_XLSX, index=False)

print(f"saved to {OUT_XLSX}")
print(result[["gps_time", "T_SR_s", "rho_SR_m"]].head())
print()
print(result["T_SR_s"].describe())


saved to T_SR.xlsx
    gps_time        T_SR_s  rho_SR_m
0  707659200  1.656140e-08  4.964983
1  707659201  1.656140e-08  4.964983
2  707659202  1.656140e-08  4.964983
3  707659203  1.656140e-08  4.964983
4  707659204  1.656140e-08  4.964983

count    8.640000e+04
mean     1.647517e-08
std      6.561618e-11
min      1.637073e-08
25%      1.641121e-08
50%      1.648416e-08
75%      1.653789e-08
max      1.656327e-08
Name: T_SR_s, dtype: float64


In [18]:
# optional preview
result.head(10)


,gps_time,dt_inst_s,d0_x,d0_y,d0_z,T_GR_MeTp_s,T_GR_TpMr_s,term1_s,term2_s,term3_s,...,d0_dot_aD_mps2,d0_dot_aC_mps2,aD_x_mps2,aD_y_mps2,aD_z_mps2,aD_mag_mps2,aC_x_mps2,aC_y_mps2,aC_z_mps2,aC_mag_mps2
0,707659200,0.00065,0.534905,0.433562,-0.725190,8.421256e-13,8.420871e-13,1.656098e-08,4.220657e-13,-2.272678e-20,...,-0.132264,0.109245,-4.732024,-4.077941,-5.746020,8.487544,-4.603380,-3.973698,-5.921836,8.488199
1,707659201,0.00065,0.535515,0.434087,-0.724425,8.421283e-13,8.420898e-13,1.656098e-08,4.220656e-13,-2.272470e-20,...,-0.132240,0.109270,-4.727052,-4.073931,-5.752991,8.487570,-4.598258,-3.969559,-5.928623,8.488224
2,707659202,0.00065,0.536126,0.434612,-0.723659,8.421310e-13,8.420925e-13,1.656098e-08,4.220656e-13,-2.272260e-20,...,-0.132216,0.109295,-4.722073,-4.069917,-5.759955,8.487596,-4.593130,-3.965416,-5.935402,8.488248
3,707659203,0.00065,0.536735,0.435136,-0.722892,8.421336e-13,8.420952e-13,1.656098e-08,4.220655e-13,-2.272051e-20,...,-0.132192,0.109320,-4.717089,-4.065897,-5.766912,8.487622,-4.587996,-3.961268,-5.942174,8.488273
4,707659204,0.00065,0.537344,0.435659,-0.722124,8.421363e-13,8.420979e-13,1.656098e-08,4.220655e-13,-2.271841e-20,...,-0.132168,0.109345,-4.712099,-4.061872,-5.773862,8.487648,-4.582856,-3.957114,-5.948938,8.488297
5,707659205,0.00065,0.537952,0.436182,-0.721355,8.421390e-13,8.421005e-13,1.656098e-08,4.220654e-13,-2.271631e-20,...,-0.132144,0.109370,-4.707102,-4.057842,-5.780805,8.487673,-4.577711,-3.952956,-5.955696,8.488322
6,707659206,0.00065,0.538560,0.436704,-0.720585,8.421416e-13,8.421032e-13,1.656098e-08,4.220654e-13,-2.271421e-20,...,-0.132119,0.109396,-4.702100,-4.053807,-5.787740,8.487699,-4.572560,-3.948793,-5.962445,8.488346
7,707659207,0.00065,0.539167,0.437226,-0.719814,8.421443e-13,8.421058e-13,1.656098e-08,4.220653e-13,-2.271210e-20,...,-0.132095,0.109421,-4.697092,-4.049767,-5.794669,8.487725,-4.567403,-3.944625,-5.969188,8.488370
8,707659208,0.00065,0.539773,0.437747,-0.719043,8.421469e-13,8.421085e-13,1.656098e-08,4.220652e-13,-2.270999e-20,...,-0.132071,0.109446,-4.692078,-4.045721,-5.801590,8.487750,-4.562240,-3.940452,-5.975923,8.488395
9,707659209,0.00065,0.540379,0.438268,-0.718270,8.421496e-13,8.421111e-13,1.656098e-08,4.220652e-13,-2.270788e-20,...,-0.132047,0.109471,-4.687059,-4.041671,-5.808504,8.487776,-4.557072,-3.936274,-5.982650,8.488419


## 输出字段说明

- `T_SR_s`：MeTp 对应的 T_SR
- `rho_SR_m`：对应距离量，`rho_SR = c_0 * T_SR`
- `dt_inst_s`：即时几何光行时
- `d0_x, d0_y, d0_z`：视线单位向量  
  \[
  d_0 = \frac{r_D-r_C}{|r_D-r_C|}
  \]
- `T_GR_MeTp_s`, `T_GR_TpMr_s`：从外部 Excel 读入的引力时延项
- `term1_s` ~ `term5_s`：原公式各分项
- `d0_dot_vD_mps`：\(d_0\cdot v_D\)
- `d0_dot_vC_mps`：\(d_0\cdot v_C\)
- `d0_dot_aD_mps2`：\(d_0\cdot a_D\)
- `d0_dot_aC_mps2`：\(d_0\cdot a_C\)
- `aD_*`：D 星加速度分量与模长，按地球点质量模型计算
- `aC_*`：C 星加速度分量与模长，按地球点质量模型计算

## 说明
这版 notebook 的核心变化只有两点：
1. **卫星角色改为 D = M，C = T**
2. **加速度改为 \(a=GM/R^2\) 的地球点质量模型**

如果你下一步还想把这版再改成“显式发射点/接收点二阶泰勒展开”的 T_SR 版本，我可以继续接着改。
